# EDA: events and anomalies

Run `make data` first. Explores the event stream the detector must recover and the irregular play the temporal model must flag.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

rows, anoms = [], []
for p in sorted(Path("../data/annotations").glob("match_*.json")):
    a = json.loads(p.read_text())
    for e in a["events"]:
        rows.append({"match": p.stem, **e})
    for x in a["anomalies"]:
        anoms.append({"match": p.stem, **x})
events, anomalies = pd.DataFrame(rows), pd.DataFrame(anoms)
print(events.event_type.value_counts(), "\n", anomalies.kind.value_counts())

## Inter-kill gaps: normal play vs. injected bursts

In [ ]:
kills = events[events.event_type == "kill"].sort_values(["match", "timestamp"])
kills["gap"] = kills.groupby("match").timestamp.diff()
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(kills.gap.dropna().clip(upper=15), bins=60)
ax.set(xlabel="seconds between consecutive kills", ylabel="count")
plt.show()

## Class balance at frame level

Banners are on screen for a small share of frames, which is why training keeps all event frames and subsamples background.

In [ ]:
dur = {"kill": 0.6, "round_start": 1.5, "round_end": 1.5}
share = events.groupby("event_type").size() * pd.Series(dur)
total = sum(
    json.loads(p.read_text())["duration"] for p in Path("../data/annotations").glob("match_*.json")
)
print((share / total).round(4))

## Anomaly durations by kind

In [ ]:
anomalies["length"] = anomalies.end - anomalies.start
anomalies.groupby("kind").length.describe()